In [ ]:
import numpy as np

from tslearn.datasets import UCR_UEA_datasets
import torch
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.linear_model import LogisticRegression
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from uni2ts.model.moirai import MoiraiModule
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

from encoder import MoiraiEncoder

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Utils

In [ ]:
PATCH_SIZES = [8, 16, 32, 64]

SIZE = "small"
DEVICE = "cuda"
NUM_VARS = 6

In [3]:
def preprocess_data(
    data: np.ndarray,
    *,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
):
    """
    data: np.ndarray of shape (N, T, V) = (n_individual, time, variate)
    Assumes NO missing values and NO padding in the raw data.
    """

    if not isinstance(data, np.ndarray):
        raise TypeError("data must be a numpy.ndarray")
    if data.ndim != 3:
        raise ValueError(f"Expected shape (N,T,V), got {data.shape}")

    N, T, V = data.shape

    # (N,T,V)
    past_target = torch.as_tensor(data, dtype=dtype, device=device)

    # observed mask: (N,T,V) all True no missing values
    past_observed_target = torch.ones((N, T, V), dtype=torch.bool, device=device)

    # padding mask: (N,T) all if no padding
    past_is_pad = torch.zeros((N, T), dtype=torch.bool, device=device)

    return past_target, past_observed_target, past_is_pad

In [4]:
def apply_pooling_pt(Z_tensor, method, num_vars=NUM_VARS):
    N, S, F = Z_tensor.shape
    P = S // num_vars # Calcul automatique du nombre de patches par variable
    
    # On reshape le tenseur pour séparer les Variables et les Patches
    # Forme résultante : (Batch, Variables, Patches, Features)
    Z_reshaped = Z_tensor.view(N, num_vars, P, F)
    
    # Basique et Global
    if method == "flatten":
        return Z_tensor.reshape(N, -1)
        
    elif method == "global_mean":
        return Z_tensor.mean(dim=1)
        
    elif method == "global_max":
        return Z_tensor.max(dim=1).values
        
    elif method == "global_min":
        return Z_tensor.min(dim=1).values
    
    elif method == "global_mean_max_min":
        return torch.cat([
            Z_tensor.mean(dim=1),
            Z_tensor.max(dim=1).values,
            Z_tensor.min(dim=1).values
        ], dim=1)

    # Pooling sur les Patches (on garde les variables distinctes) ---
    # Réduction sur la dimension 2 (Patches). Résultat : (N, num_vars, F), puis on aplatit
    elif method == "mean_over_patches":
        return Z_reshaped.mean(dim=2).reshape(N, -1)
        
    elif method == "max_over_patches":
        return Z_reshaped.max(dim=2).values.reshape(N, -1)
        
    elif method == "min_over_patches":
        return Z_reshaped.min(dim=2).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_patches":
        p_mean = Z_reshaped.mean(dim=2).reshape(N, -1)
        p_max  = Z_reshaped.max(dim=2).values.reshape(N, -1)
        p_min  = Z_reshaped.min(dim=2).values.reshape(N, -1)
        return torch.cat([p_mean, p_max, p_min], dim=1)

    # Pooling sur les Variables (on synchronise les patches entre variables) ---
    # Réduction sur la dimension 1 (Variables). Résultat : (N, P, F), puis on aplatit
    elif method == "mean_over_variables":
        return Z_reshaped.mean(dim=1).reshape(N, -1)
        
    elif method == "max_over_variables":
        return Z_reshaped.max(dim=1).values.reshape(N, -1)
        
    elif method == "min_over_variables":
        return Z_reshaped.min(dim=1).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_variables":
        v_mean = Z_reshaped.mean(dim=1).reshape(N, -1)
        v_max  = Z_reshaped.max(dim=1).values.reshape(N, -1)
        v_min  = Z_reshaped.min(dim=1).values.reshape(N, -1)
        return torch.cat([v_mean, v_max, v_min], dim=1)

    else:
        raise ValueError(f"Method {method} unknow")

mask_pooling_methods = [
    "flatten", 
    "mean_over_variables", 
    "max_over_variables", 
    "min_over_variables",
    "mean_max_min_over_variables"
]

In [5]:
alphas_to_test = [0.1,1,10,30,100,300,1000,10000]

models = {
    'Ridge': RidgeClassifierCV(alphas=alphas_to_test, cv=3),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# Data

Import

In [6]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

NUM_CLASSES = len(torch.unique(y_train_tensor))

Prepare data

In [7]:
X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = (
    preprocess_data(X_train, device=DEVICE)
)

X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = (
    preprocess_data(X_test, device=DEVICE)
)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

num_classes = len(torch.unique(y_train_tensor))
NUM_VARS = 6

print(X_train_target_tensor.shape)
print(X_train_is_target_observed.shape)
print(X_train_is_target_padded.shape)

torch.Size([2459, 36, 6])
torch.Size([2459, 36, 6])
torch.Size([2459, 36])


Creat Z_train and Z_test for each patch size

In [8]:
Z_train_mask_patch = {}
Z_test_mask_patch = {}
for patch in PATCH_SIZES:
    
    # 1. Définir les paramètres et l'encodeur pour la taille de patch actuelle
    MODEL_PARAMS = {
        "patch_size": patch,
        "num_samples": 100,
        "target_dim": 1,
        "feat_dynamic_real_dim": 5,
        "past_feat_dynamic_real_dim": 0,
    }
    
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=patch,
        context_length=36,
        **MODEL_PARAMS,
    )
    encoder.eval()
    encoder.to(DEVICE)
    
    # 2. Extraire les embeddings (Z_train et Z_test)
    with torch.no_grad():
        Z_train = encoder(
            past_target=X_train_target_tensor,
            past_observed_target=X_train_is_target_observed,
            past_is_pad=X_train_is_target_padded,
        )
        Z_test = encoder(
            past_target=X_test_target_tensor,
            past_observed_target=X_test_is_target_observed,
            past_is_pad=X_test_is_target_padded,
        )
    
    N, S, F = Z_train.shape
    N_test = Z_test.shape[0]
    
    # Calcul du nombre de patchs par variable (Contexte + 1 Masque)
    P = S // NUM_VARS  
    
    # 1. Reshape : (Batch, Variables, Patchs, Features)
    Z_train_reshaped = Z_train.view(N, NUM_VARS, P, F)
    Z_test_reshaped = Z_test.view(N_test, NUM_VARS, P, F)
    
    # 2. On isole UNIQUEMENT le masque (le dernier patch de la dimension 2)
    # On utilise -1: pour garder la dimension (taille 1) et rester compatible avec ta fonction de pooling
    Z_train_mask_only = Z_train_reshaped[:, :, -1:, :]
    Z_test_mask_only = Z_test_reshaped[:, :, -1:, :]
    
    # 3. On reforme le tenseur (Batch, Variables * 1_Masque, Features)
    # La nouvelle séquence S fera exactement la taille de NUM_VARS (soit 6)
    Z_train_mask_patch[patch] = Z_train_mask_only.reshape(N, -1, F)
    Z_test_mask_patch[patch] = Z_test_mask_only.reshape(N_test, -1, F)
        

In [9]:

print(f"{'Patch Size':<12} | {'Train Shape':<25} | {'Test Shape':<25}")
for patch in PATCH_SIZES:
    train_tensor = Z_train_mask_patch[patch]
    test_tensor = Z_test_mask_patch[patch]
    print(f"{patch:<12} | {str(list(train_tensor.shape)):<25} | {str(list(test_tensor.shape)):<25}")

Patch Size   | Train Shape               | Test Shape               
8            | [2459, 6, 384]            | [2466, 6, 384]           
16           | [2459, 6, 384]            | [2466, 6, 384]           
32           | [2459, 6, 384]            | [2466, 6, 384]           
64           | [2459, 6, 384]            | [2466, 6, 384]           


# Logistic regression and random forest

### Training

We creat a dataframe to stock the results and we fill it

We train

In [10]:
df_mask_ridge = pd.DataFrame(index=mask_pooling_methods, columns=PATCH_SIZES)
df_mask_rf = pd.DataFrame(index=mask_pooling_methods, columns=PATCH_SIZES)
df_mask_ridge.index.name = "Pooling (Mask Only)"
df_mask_rf.index.name = "Pooling (Mask Only)"

for patch in PATCH_SIZES:
    Z_train = Z_train_mask_patch[patch]
    Z_test = Z_test_mask_patch[patch]
    
    for pool in mask_pooling_methods:
        X_train_pool = apply_pooling_pt(Z_train, pool, num_vars=NUM_VARS).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool, num_vars=NUM_VARS).cpu().numpy()

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_pool)
        X_test_scaled = scaler.transform(X_test_pool)
        
        clf_ridge = models['Ridge']
        clf_ridge.fit(X_train_scaled, y_train)
        df_mask_ridge.loc[pool, patch] = round(accuracy_score(y_test, clf_ridge.predict(X_test_scaled)), 3)
        
        clf_rf = models['RandomForest']
        clf_rf.fit(X_train_scaled, y_train)
        df_mask_rf.loc[pool, patch] = round(accuracy_score(y_test, clf_rf.predict(X_test_scaled)), 3)

### Results

Ridge

In [11]:
display(df_mask_ridge.astype(float))

,8,16,32,64
Pooling (Mask Only),,,,
flatten,0.588,0.591,0.601,0.601
mean_over_variables,0.567,0.564,0.571,0.577
max_over_variables,0.524,0.540,0.541,0.567
min_over_variables,0.535,0.537,0.536,0.564
mean_max_min_over_variables,0.568,0.567,0.565,0.583


RF

In [12]:
display(df_mask_rf.astype(float))

,8,16,32,64
Pooling (Mask Only),,,,
flatten,0.521,0.524,0.536,0.567
mean_over_variables,0.537,0.530,0.534,0.560
max_over_variables,0.508,0.505,0.501,0.555
min_over_variables,0.498,0.505,0.511,0.549
mean_max_min_over_variables,0.536,0.526,0.530,0.549


## Multi Patch

### Training

In [13]:
pool_method = "flatten"

X_train_multi_mask = []
X_test_multi_mask = []

for patch in PATCH_SIZES:
    Z_train = Z_train_mask_patch[patch]
    Z_test = Z_test_mask_patch[patch]
    
    X_train_pool = apply_pooling_pt(Z_train, pool_method, num_vars=NUM_VARS).cpu().numpy()
    X_test_pool = apply_pooling_pt(Z_test, pool_method, num_vars=NUM_VARS).cpu().numpy()
    
    X_train_multi_mask.append(X_train_pool)
    X_test_multi_mask.append(X_test_pool)

X_train_combined = np.concatenate(X_train_multi_mask, axis=1)
X_test_combined = np.concatenate(X_test_multi_mask, axis=1)

scaler = StandardScaler()
X_train_combined_scaled = scaler.fit_transform(X_train_combined)
X_test_combined_scaled = scaler.transform(X_test_combined)



results = {}
for model_name, clf in models.items():
    clf.fit(X_train_combined_scaled, y_train)
    y_pred = clf.predict(X_test_combined_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[model_name] = round(acc, 3)


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=4.50345e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=4.74556e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=4.60111e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)


### Results

In [14]:
for model_name, acc in results.items():
    print(f"Modèle : {model_name:<15} | Test Accuracy : {acc:.3f}")

Modèle : Ridge           | Test Accuracy : 0.633
Modèle : RandomForest    | Test Accuracy : 0.573
